# 01 · Extraction and cleaning

Goal: show that every supported file type is read **with its location** (page/slide or heading section), that tables stay as aligned rows, and that cleaning removes noise (running headers, page numbers, broken hyphenation) without touching real content or Urdu text.

In [1]:
import sys, uuid, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "backend"))
from tests.fixtures.samples import build_samples
SAMPLES = build_samples(ROOT / "data" / "samples")
for name, path in SAMPLES.items():
    print(f"{name:14s} {path.name:22s} {path.stat().st_size:>7} bytes")

policy_pdf     leave_policy.pdf          6780 bytes
scanned_pdf    scanned.pdf            1506536 bytes
handbook_docx  it_security.docx         36767 bytes
slides_pptx    onboarding.pptx          29385 bytes
notes_md       faq.md                     128 bytes
urdu_txt       urdu_notice.txt            139 bytes
empty_pdf      empty.pdf                    0 bytes
fake_pdf       invoice.pdf                204 bytes


## 1. PDF: raw extraction
Each unit is a paragraph or table with its page and heading path. Note the running header and the page-number footer on every page.

In [2]:
from app.ingestion.extractors.pdf import extract_pdf
raw = extract_pdf(SAMPLES["policy_pdf"].read_bytes(), max_pages=300)
print("pages:", raw.page_count)
for u in raw.units:
    print(f"p{u.page} [{u.kind:5s}] section={u.section!r:22s} | {u.text[:70]!r}")

pages: 3
p1 [text ] section=None                   | 'Acme Corp - Employee Handbook 2026'
p1 [text ] section='1. Annual Leave'      | 'Full-time employees receive 20 days of paid annual leave per calendar '
p1 [text ] section='1. Annual Leave'      | 'Page 1 of 3'
p2 [text ] section='1. Annual Leave'      | 'Acme Corp - Employee Handbook 2026'
p2 [text ] section='2. Sick Leave'        | 'Employees are entitled to 10 paid sick days per year. A medical certif'
p2 [table] section='2. Sick Leave'        | '| Leave type | Days per year | Carry over |\n| --- | --- | --- |\n| Annu'
p2 [text ] section='2. Sick Leave'        | 'Page 2 of 3'
p3 [text ] section='2. Sick Leave'        | 'Acme Corp - Employee Handbook 2026'
p3 [text ] section='3. Remote Work'       | 'Staff may work remotely up to two days per week with manager approval.'
p3 [text ] section='3. Remote Work'       | 'Page 3 of 3'


## 2. Cleaning
Headers repeated on ≥60% of pages and page-number lines are dropped; `por-\ntal` becomes `portal`; single line breaks inside paragraphs become spaces.

In [3]:
from app.ingestion.cleaning import clean_units, text_stats
clean = clean_units(raw.units, raw.page_count)
print("before:", dict(text_stats(raw.units)))
print("after: ", dict(text_stats(clean)))
removed = {u.text for u in raw.units} - {u.text for u in raw.units if any(u.text.split()[0] in c.text for c in clean)}
print("\nremoved lines:", sorted(removed))
print("\nhyphen repaired:", "HR portal" in " ".join(u.text for u in clean))

before: {'text_units': 9, 'text_chars': 677, 'table_units': 1, 'table_chars': 131}
after:  {'text_units': 3, 'text_chars': 540, 'table_units': 1, 'table_chars': 131}

removed lines: ['Acme Corp - Employee Handbook 2026', 'Page 1 of 3', 'Page 2 of 3', 'Page 3 of 3']

hyphen repaired: True


## 3. Tables become Markdown rows
Cells are read from their own rectangle on the page, so words keep their spaces.

In [4]:
print(next(u for u in clean if u.kind == "table").text)

| Leave type | Days per year | Carry over |
| --- | --- | --- |
| Annual | 20 | Up to 5 |
| Sick | 10 | No |
| Parental | 90 | No |


## 4. DOCX: headings give the section path (DOCX has no fixed pages)

In [5]:
from app.ingestion.extractors.office import extract_docx, extract_pptx
for u in extract_docx(SAMPLES["handbook_docx"].read_bytes()).units:
    print(f"[{u.kind:5s}] {u.section!r}\n        {u.text[:80]!r}")

[text ] 'IT Security Handbook > Passwords'
        'Passwords must be at least 14 characters long and changed every 180 days. Passwo'
[text ] 'IT Security Handbook > Passwords > Multi-factor authentication'
        'MFA is mandatory for email, VPN and the HR portal.'
[text ] 'IT Security Handbook > Devices'
        'Company laptops must use full-disk encryption.'
[table] 'IT Security Handbook > Devices'
        '| Device | Encryption |\n| --- | --- |\n| Laptop | BitLocker |\n| Phone | Built-in '


## 5. PPTX, Markdown and Urdu text (bonus file types)

In [6]:
from app.ingestion.extractors.text import extract_text
for u in extract_pptx(SAMPLES["slides_pptx"].read_bytes(), max_pages=300).units:
    print(f"slide {u.page} section={u.section!r}: {u.text!r}")
for u in extract_text(SAMPLES["notes_md"].read_bytes(), markdown=True).units:
    print(f"md section={u.section!r}: {u.text!r}")
for u in clean_units(extract_text(SAMPLES["urdu_txt"].read_bytes(), markdown=False).units, None):
    print("urdu:", u.text)

slide 1 section='Onboarding Week': 'Day 1: laptop setup\nDay 2: security training'
slide 2 section='Benefits': 'Health insurance starts on the first day of employment.'
slide 2 section='Benefits': 'Dental cover is optional.'
md section='FAQ > Payroll': 'Salaries are paid on the last working day of each month.'
md section='FAQ > Expenses': 'Submit expense claims within 30 days.'
urdu: دفتر جمعہ کو دوپہر ایک بجے بند ہو جائے گا۔
urdu: تمام ملازمین اپنی حاضری درج کریں۔


## 6. Bad uploads are rejected with the right status code

In [7]:
from app.errors import AppError
from app.ingestion.pipeline import process_file
cases = {"empty.pdf": SAMPLES["empty_pdf"], "invoice.pdf (renamed .exe)": SAMPLES["fake_pdf"],
         "scanned.pdf": SAMPLES["scanned_pdf"], "setup.exe": SAMPLES["fake_pdf"]}
for label, path in cases.items():
    name = label.split(" ")[0]
    try:
        process_file(name, path.read_bytes(), document_id=str(uuid.uuid4()))
        print(f"{label:28s} accepted (unexpected)")
    except AppError as e:
        print(f"{label:28s} -> {e.status_code} {e.code}: {e.message}")

empty.pdf                    -> 400 EMPTY_FILE: The uploaded file is empty.
invoice.pdf (renamed .exe)   -> 415 UNSUPPORTED_FILE_TYPE: File content is not a valid PDF.


scanned.pdf                  -> 400 NO_EXTRACTABLE_TEXT: No extractable text found. The file may be scanned images or empty.
setup.exe                    -> 415 UNSUPPORTED_FILE_TYPE: Unsupported file type '.exe'.
